## EcoHome Energy Advisor - RAG Setup

In this notebook, you'll set up the Retrieval-Augmented Generation (RAG) pipeline for the EcoHome Energy Advisor. This will allow the agent to access and cite relevant energy-saving tips and best practices.

## Learning Objectives
- Set up ChromaDB vector store
- Load and process energy-saving documents
- Create embeddings for document chunks
- Implement semantic search functionality
- Test the RAG pipeline

## Documents Available
- `tip_device_best_practices.txt` - Device-specific optimization tips
- `tip_energy_savings.txt` - General energy-saving strategies
- `tip_hvac_optimization.txt` - HVAC optimization strategies
- `tip_smart_home_automation.txt` - Smart home automation tips
- `tip_renewable_energy_integration.txt` - Renewable energy integration
- `tip_seasonal_energy_management.txt` - Seasonal energy management
- `tip_energy_storage_optimization.txt` - Energy storage optimization

## 1. Import Required Libraries


In [1]:
# Import the necessary libraries for RAG setup
import os
from langchain_chroma  import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

## 2. Load and Process Documents


In [3]:
# Load the energy-saving tip documents
# Load all tip documents in data/documents
# Use TextLoader to load the documents

documents = []
document_paths = [
    "data/documents/tip_device_best_practices.txt",
    "data/documents/tip_energy_savings.txt",
    "data/documents/tip_hvac_optimization.txt",
    "data/documents/tip_smart_home_automation.txt",
    "data/documents/tip_renewable_energy_integration.txt",
    "data/documents/tip_seasonal_energy_management.txt",
    "data/documents/tip_energy_storage_optimization.txt",
]

for doc_path in document_paths:
    if os.path.exists(doc_path):
        loader = TextLoader(doc_path)
        docs = loader.load()
        documents.extend(docs)
        print(f"Loaded {len(docs)} documents from {doc_path}")
    else:
        print(f"Warning: {doc_path} not found")

print(f"Total documents loaded: {len(documents)}")


Loaded 1 documents from data/documents/tip_device_best_practices.txt
Loaded 1 documents from data/documents/tip_energy_savings.txt
Loaded 1 documents from data/documents/tip_hvac_optimization.txt
Loaded 1 documents from data/documents/tip_smart_home_automation.txt
Loaded 1 documents from data/documents/tip_renewable_energy_integration.txt
Loaded 1 documents from data/documents/tip_seasonal_energy_management.txt
Loaded 1 documents from data/documents/tip_energy_storage_optimization.txt
Total documents loaded: 7


## 3. Split Documents into Chunks


In [4]:
# Split documents into smaller chunks for better retrieval
# Use RecursiveCharacterTextSplitter with appropriate chunk_size and chunk_overlap
# Experiment with different chunk sizes (e.g., 500, 1000, 1500 characters)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, # [1000, 200]
    chunk_overlap=200,  # [200, 50]
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

# Split the documents
splits = text_splitter.split_documents(documents)
print(f"Split {len(documents)} documents into {len(splits)} chunks")

# Show sample chunk
if splits:
    print(f"\nSample chunk (first 200 characters):")
    print(splits[0].page_content[:200] + "...")


Split 7 documents into 14 chunks

Sample chunk (first 200 characters):
Large devices like electric vehicles, washing machines and dishwashers often support delayed start or timer functions. Schedule these devices to run outside of peak electricity pricing hours or during...


## 4. Create Vector Store


In [5]:
# Create a ChromaDB vector store
# Initialize OpenAIEmbeddings
# Create the vector store with the document chunks
# Persist the vector store to disk for future use
import shutil

# Set up the persist directory
persist_directory = "data/vectorstore"

# Rebuild from scratch: Chroma.from_documents appends to an existing store,
# which would duplicate chunks, and tools.py only builds the store when it is missing.
if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)
os.makedirs(persist_directory, exist_ok=True)

# Initialize embeddings
embeddings = OpenAIEmbeddings(
    base_url="https://openai.vocareum.com/v1",
    # base_url=os.getenv("OPENAI_API_BASE_URL"),
    api_key=os.getenv("VOCAREUM_API_KEY")
    # api_key=os.getenv("OPENAI_API_KEY")
)

# Create the vector store
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory=persist_directory
)

print(f"Vector store created and persisted to {persist_directory}")
print(f"Total vectors stored: {len(splits)}")

Vector store created and persisted to data/vectorstore
Total vectors stored: 14


## 5. Test the RAG Pipeline


In [6]:
# Test the search functionality
# Try different queries related to energy optimization
# Test queries like:
# - "electric vehicle charging tips"
# - "thermostat optimization"
# - "dishwasher energy saving"
# - "solar power maximization"

test_queries = [
    "electric vehicle charging tips",
    "thermostat optimization",
    "dishwasher energy saving",
    "solar power maximization",
    "HVAC system efficiency",
    "pool pump scheduling",
    # Queries for the added documents
    "pre-cooling before peak pricing hours",
    "smart plug standby power automation",
    "using surplus solar generation at home",
    "winter energy saving tips",
    "home battery charging and discharging strategy",
]

print("=== Testing Vector Search ===")
for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("===")
    docs = vectorstore.similarity_search(query, k=2)
    for i, doc in enumerate(docs):
        print(f"  Result {i+1}: {doc.page_content[:400]}...")
        print(f"  Source: {doc.metadata.get('source', 'unknown')}")
        print("===")

# 最も意味が近いチャンクのトップ2を表示している

=== Testing Vector Search ===

Query: 'electric vehicle charging tips'
===
  Result 1: Battery Health:
- Keep a reserve of 10-20% charge to protect battery life and provide backup power
- Avoid frequent full discharges when possible
- Install the battery in a location with moderate temperatures, as extreme heat or cold reduces performance
- Keep battery firmware up to date for improved control algorithms

Backup Power Planning:
- Increase the backup reserve before storms or forecast...
  Source: data/documents/tip_energy_storage_optimization.txt
===
  Result 2: Large devices like electric vehicles, washing machines and dishwashers often support delayed start or timer functions. Schedule these devices to run outside of peak electricity pricing hours or during periods of high solar generation. Consult your device manual for specific scheduling instructions.

Electric Vehicle Charging:
- Charge during off-peak hours (typically 11 PM to 6 AM) to save money
-...
  Source: data/documents/tip

## 6. Test the Search Tool


In [7]:
# Test the search_energy_tips tool from tools.py
# Import and test the tool with various queries
# Verify that it returns relevant results

from tools import search_energy_tips

# Test the search_energy_tips function
print("=== Testing search_energy_tips Tool ===")

test_queries = [
    "electric vehicle charging",
    "thermostat settings",
    "dishwasher optimization",
    "solar power tips",
    "heat pump maintenance",
    "battery backup reserve before storms",
]

for query in test_queries:
    print("===")
    print(f"\nQuery: '{query}'")
    print("===")
    result = search_energy_tips.invoke(
        input={
            "query": query, 
            "max_results": 3,
        }
    )
    
    if "error" in result:
        print(f"  Error: {result['error']}")
    else:
        print(f"  Found {result['total_results']} results")
        for i, tip in enumerate(result['tips']):
            print(f"    {i+1}. {tip['content'][:400]}...")
            print(f"       Source: {tip['source']}")
            print(f"       Relevance: {tip['relevance_score']}")


=== Testing search_energy_tips Tool ===
===

Query: 'electric vehicle charging'
===
  Found 3 results
    1. Large devices like electric vehicles, washing machines and dishwashers often support delayed start or timer functions. Schedule these devices to run outside of peak electricity pricing hours or during periods of high solar generation. Consult your device manual for specific scheduling instructions.

Electric Vehicle Charging:
- Charge during off-peak hours (typically 11 PM to 6 AM) to save money
-...
       Source: data/documents/tip_device_best_practices.txt
       Relevance: high
    2. Battery Health:
- Keep a reserve of 10-20% charge to protect battery life and provide backup power
- Avoid frequent full discharges when possible
- Install the battery in a location with moderate temperatures, as extreme heat or cold reduces performance
- Keep battery firmware up to date for improved control algorithms

Backup Power Planning:
- Increase the backup reserve before storms or forec